## Magic methods
- Em Python, os métodos especiais (também chamados de magic methods ou dunder methods) são funções cujo nome começa e termina com dois sublinhados (__). Eles permitem que objetos se comportem como tipos nativos (listas, dicionários, números, etc.) e são muito úteis em análises de dados para criar objetos que interajam naturalmente com bibliotecas como pandas, numpy e matplotlib.

Abaixo estão os métodos mais usados por analistas de dados:

|Método	|Quando usar	|Por que é útil|
|:---|---|---|
| __init __	|Instanciação de objeto	|Inicializa atributos (ex.: dados, metadados)|
|__repr __	|Impressão em REPL	|Representação “oficial” do objeto|
|__str __	|Impressão amigável	|Representação legível para o usuário|
|__len __	|en(obj)	|Retorna tamanho (ex.: número de linhas)|
|__getitem __	|obj[key]	|Indexação e slicing customizados|
|__setitem __	|obj[key] = value	|Atribuição por índice|
|__iter __	|for x in obj	|Iteração sobre linhas ou elementos|
|__next __	|next(obj)	|Complementa __iter__|
|__contains __	|key in obj	|Verifica presença de chave/valor|
|__add __, __sub __, __mul __, __truediv __	|Operações aritméticas	|Soma, subtração, etc., sobre objetos de dados|
|__eq __, __lt __, __gt __	|Comparações	|Permite ordenar ou filtrar objetos|
|__call __	|obj()	|Faz o objeto “executável” (ex.: função de transformação)|
|__enter __, __exit __	|with statement	|Gerenciamento de contexto (ex.: abrir/fechar arquivos)|

In [ ]:
#Exemplo de Código: Uma Classe Dataset Customizada

import pandas as pd

class Dataset:
    """
    Um wrapper simples para DataFrames que demonstra métodos especiais.
    """

    def __init__(self, data, name="dataset"):
        """
        Construtor: recebe dados (pode ser dict, CSV path, etc.)
        """
        if isinstance(data, str):
            # Assume que é caminho para CSV
            self.df = pd.read_csv(data)
        else:
            self.df = pd.DataFrame(data)
        self.name = name

    # ---------------------------------------
    # 1. Representação legível
    # ---------------------------------------
    def __repr__(self):
        return f"<Dataset '{self.name}' com {len(self)} linhas>"

    # ---------------------------------------
    # 2. Representação amigável
    # ---------------------------------------
    def __str__(self):
        return f"Dataset '{self.name}':\n{self.df.head()}"

    # ---------------------------------------
    # 3. Tamanho (len(dataset))
    # ---------------------------------------
    def __len__(self):
        return len(self.df)

    # ---------------------------------------
    # 4. Indexação (dataset['col'])
    # ---------------------------------------
    def __getitem__(self, key):
        return self.df[key]

    # ---------------------------------------
    # 5. Atribuição por índice (dataset['col'] = ...)
    # ---------------------------------------
    def __setitem__(self, key, value):
        self.df[key] = value

    # ---------------------------------------
    # 6. Iteração (for row in dataset)
    # ---------------------------------------
    def __iter__(self):
        # devolve um iterador de Series (linhas)
        return self.df.iterrows()

    # ---------------------------------------
    # 7. Operação aritmética (dataset + outro)
    # ---------------------------------------
    def __add__(self, other):
        if isinstance(other, Dataset):
            # concatena verticalmente (um em cima do outro)
            new_df = pd.concat([self.df, other.df], ignore_index=True)
            return Dataset(new_df, name=f"{self.name}+{other.name}")
        raise TypeError("Só é possível somar com outro Dataset")

    # ---------------------------------------
    # 8. Comparação (dataset == outro)
    # ---------------------------------------
    def __eq__(self, other):
        if isinstance(other, Dataset):
            return self.df.equals(other.df)
        return False

    # ---------------------------------------
    # 9. Função de transformação (dataset())
    # ---------------------------------------
    def __call__(self, func, *args, **kwargs):
        """
        Permite aplicar uma função ao DataFrame interno.
        Ex.: ds(lambda df: df.describe())
        """
        return func(self.df, *args, **kwargs)

In [ ]:
# Como usar

# 1. Cria um dataset a partir de um CSV
ds1 = Dataset("dados.csv", name="sales")

# 2. Imprime representação oficial
print(ds1)          # <Dataset 'sales' com 150 linhas>

# 3. Imprime cabeçalho amigável
print(str(ds1))

# 4. Tamanho
print(len(ds1))     # 150

# 5. Indexação
print(ds1["produto"])

# 6. Atribuição
ds1["nova_coluna"] = ds1["quantidade"] * ds1["preço"]

# 7. Iteração
for idx, row in ds1:
    print(idx, row["produto"])

# 8. Soma de datasets
ds2 = Dataset({"produto": ["A", "B"], "quantidade": [10, 20]})
ds_total = ds1 + ds2

# 9. Função de transformação
print(ds1(lambda df: df.describe()))

### Detalhes do Código

1. __init__

- Detecta se a entrada é um caminho de arquivo (str) ou dados estruturados e cria um pandas.DataFrame.
- Armazena o nome para facilitar a identificação.

2. __repr__

- Retorna uma string curta que mostra o nome e o número de linhas, útil em notebooks.

3. __str__

- Exibe as primeiras linhas do DataFrame, facilitando a visualização rápida.

4. __len__

- Permite usar len(ds1) para obter o número de linhas.

5. __getitem__ / __setitem__

- Permitem acessar e modificar colunas como se o objeto fosse um dicionário.

6. __iter__

- Converte cada linha em uma tupla (index, Series).
- Facilita loops for.

7. __add__

- Sobrescreve + para concatenar dois Dataset verticalmente, retornando um novo objeto.

8. __eq__

- Define igualdade baseada na equivalência de DataFrames (df.equals).

9. __call__

- Faz o objeto “executável”; recebe uma função que opera sobre o DataFrame interno.
- Útil para aplicar rapidamente transformações sem acessar ds1.df.

---

## Exemplo de Self e outras coisas a mais

Em Python, a palavra‑chave self é usada em métodos de uma classe para referenciar a própria instância que está executando o método.

- Instância: É um objeto concreto criado a partir de uma classe. Cada instância possui seus próprios atributos (dados) e pode chamar os métodos da classe.
- self permite que o método acesse e modifique esses atributos. Sem self, o método não saberia a qual objeto pertence.
- O nome self não é obrigatório; você pode usar qualquer nome, mas por convenção todos os desenvolvedores usam self.

In [1]:
class Pessoa:
    # Atributo de classe – comum a todas as instâncias
    especie = "Homo sapiens"

    def __init__(self, nome, idade):
        """
        Construtor: chamado quando criamos uma nova Pessoa.
        `self` referencia a instância que está sendo criada.
        """
        self.nome = nome   # atributo de instância
        self.idade = idade # atributo de instância

    def apresentar(self):
        """
        Método que usa `self` para acessar atributos da própria instância.
        """
        return f"Olá, eu sou {self.nome}, tenho {self.idade} anos."

    @classmethod
    def especie_global(cls):
        """
        Método de classe: recebe `cls` (classe) em vez de `self`.
        Útil para acessar atributos de classe.
        """
        return f"Todos nós somos {cls.especie}."

# Criando instâncias
p1 = Pessoa("Ana", 30)
p2 = Pessoa("Bruno", 25)

print(p1.apresentar())  # -> Olá, eu sou Ana, tenho 30 anos.
print(p2.apresentar())  # -> Olá, eu sou Bruno, tenho 25 anos.
print(Pessoa.especie_global())  # -> Todos nós somos Homo sapiens.

Olá, eu sou Ana, tenho 30 anos.
Olá, eu sou Bruno, tenho 25 anos.
Todos nós somos Homo sapiens.


Linha|	O que acontece|	Por que é importante
|---|---|---
class Pessoa:|	Define a classe Pessoa.|	Toda a estrutura de dados e comportamento estão aqui.
especie = "Homo sapiens"|	Atributo de classe.|	Compartilhado por todas as instâncias.
def __init__(self, nome, idade):	Construtor.	Executado quando Pessoa(...) é chamado.
self.nome = nome / self.idade = idade|	Cria atributos de instância.|	Cada objeto tem seu próprio nome e idade.
def apresentar(self):|	Método de instância.|	self permite acessar self.nome e self.idade.
return f"Olá, eu sou {self.nome}, tenho {self.idade} anos."|	Retorna string formatada.|	Mostra como self é usado em expressões.
@classmethod / def especie_global(cls):|	Método de classe.|	Usa cls para acessar atributos de classe, não de instância.
p1 = Pessoa("Ana", 30)|	Cria uma nova instância.|	__init__ é chamado, self aponta para p1.
p1.apresentar()|	Chama método da instância.|	self dentro de apresentar aponta para p1.
Pessoa.especie_global()|	Chama método de classe.|	cls aponta para a classe Pessoa.